In [35]:
import mlflow
import pandas as pd
import mlflow.sklearn
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import re
import string
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
import numpy as np
import nltk

In [36]:
df = pd.read_csv('IMDB.csv')
df = df.sample(500)
df.to_csv('data.csv', index=False)
df.head()

,review,sentiment
736,"I went to see Random Hearts with 3 friends, an...",negative
944,"""In Love and War"" is a simple feel-good TV-fil...",positive
575,This third Darkman was definitely better than ...,negative
578,NOTHING (3+ outta 5 stars) Another weird premi...,positive
697,So you think a talking parrot is not your cup ...,positive


In [37]:
# data preprocessing

# Define text preprocessing functions
def lemmatization(text):
    """Lemmatize the text."""
    lemmatizer = WordNetLemmatizer()
    text = text.split()
    text = [lemmatizer.lemmatize(word) for word in text]
    return " ".join(text)

def remove_stop_words(text):
    """Remove stop words from the text."""
    stop_words = set(stopwords.words("english"))
    text = [word for word in str(text).split() if word not in stop_words]
    return " ".join(text)

def removing_numbers(text):
    """Remove numbers from the text."""
    text = ''.join([char for char in text if not char.isdigit()])
    return text

def lower_case(text):
    """Convert text to lower case."""
    text = text.split()
    text = [word.lower() for word in text]
    return " ".join(text)

def removing_punctuations(text):
    """Remove punctuations from the text."""
    text = re.sub('[%s]' % re.escape(string.punctuation), ' ', text)
    text = text.replace('؛', "")
    text = re.sub('\s+', ' ', text).strip()
    return text

def removing_urls(text):
    """Remove URLs from the text."""
    url_pattern = re.compile(r'https?://\S+|www\.\S+')
    return url_pattern.sub(r'', text)

def normalize_text(df):
    """Normalize the text data."""
    try:
        df['review'] = df['review'].apply(lower_case)
        df['review'] = df['review'].apply(remove_stop_words)
        df['review'] = df['review'].apply(removing_numbers)
        df['review'] = df['review'].apply(removing_punctuations)
        df['review'] = df['review'].apply(removing_urls)
        df['review'] = df['review'].apply(lemmatization)
        return df
    except Exception as e:
        print(f'Error during text normalization: {e}')
        raise

In [38]:
nltk.download('wordnet')
df = normalize_text(df)
df.head()

[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\anish\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


,review,sentiment
736,went see random heart friend first thought may...,negative
944,in love war simple feel good tv film viewed su...,positive
575,third darkman definitely better second one sti...,negative
578,nothing outta star another weird premise direc...,positive
697,think talking parrot cup tea huh well think ag...,positive


In [39]:
df['sentiment'].value_counts()

sentiment
negative    251
positive    249
Name: count, dtype: int64

In [40]:
x = df['sentiment'].isin(['positive','negative'])
df = df[x]

In [41]:
df['sentiment'] = df['sentiment'].map({'positive':1, 'negative':0})
df.head()

,review,sentiment
736,went see random heart friend first thought may...,0
944,in love war simple feel good tv film viewed su...,1
575,third darkman definitely better second one sti...,0
578,nothing outta star another weird premise direc...,1
697,think talking parrot cup tea huh well think ag...,1


In [42]:
df.isnull().sum()

review       0
sentiment    0
dtype: int64

In [43]:
vectorizer = CountVectorizer(max_features=100)
X = vectorizer.fit_transform(df['review'])
y = df['sentiment']

In [44]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

In [45]:
!pip install dagshub

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
import dagshub
import os
from dotenv import load_dotenv
load_dotenv()
import logging

for logger_name in ["dagshub", "mlflow", "urllib3", "httpx"]:
    logging.getLogger(logger_name).setLevel(logging.WARNING)

load_dotenv()

# Initialize DagsHub tracking without hardcoded values
dagshub.init(
    repo_owner=os.getenv("DAGSHUB_REPO_OWNER"),
    repo_name=os.getenv("DAGSHUB_REPO_NAME"),
    mlflow=True,
)
mlflow.set_experiment("Logistic Regression Baseline")


In [47]:
import mlflow
import logging
import os
import time
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# Configure logging
logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")

logging.info("Starting MLflow run...")

with mlflow.start_run():
    start_time = time.time()
    
    try:
        logging.info("Logging preprocessing parameters...")
        mlflow.log_param("vectorizer", "Bag of Words")
        mlflow.log_param("num_features", 100)
        mlflow.log_param("test_size", 0.25)

        logging.info("Initializing Logistic Regression model...")
        model = LogisticRegression(max_iter=1000)  # Increase max_iter to prevent non-convergence issues

        logging.info("Fitting the model...")
        model.fit(X_train, y_train)
        logging.info("Model training complete.")

        logging.info("Logging model parameters...")
        mlflow.log_param("model", "Logistic Regression")

        logging.info("Making predictions...")
        y_pred = model.predict(X_test)

        logging.info("Calculating evaluation metrics...")
        accuracy = accuracy_score(y_test, y_pred)
        precision = precision_score(y_test, y_pred)
        recall = recall_score(y_test, y_pred)
        f1 = f1_score(y_test, y_pred)

        logging.info("Logging evaluation metrics...")
        mlflow.log_metric("accuracy", accuracy)
        mlflow.log_metric("precision", precision)
        mlflow.log_metric("recall", recall)
        mlflow.log_metric("f1_score", f1)

        logging.info("Saving and logging the model...")
        mlflow.sklearn.log_model(model, "model")

        # Log execution time
        end_time = time.time()
        logging.info(f"Model training and logging completed in {end_time - start_time:.2f} seconds.")

        # Save and log the notebook
        # notebook_path = "exp1_baseline_model.ipynb"
        # logging.info("Executing Jupyter Notebook. This may take a while...")
        # os.system(f"jupyter nbconvert --to notebook --execute --inplace {notebook_path}")
        # mlflow.log_artifact(notebook_path)

        # logging.info("Notebook execution and logging complete.")

        # Print the results for verification
        logging.info(f"Accuracy: {accuracy}")
        logging.info(f"Precision: {precision}")
        logging.info(f"Recall: {recall}")
        logging.info(f"F1 Score: {f1}")

    except Exception as e:
        logging.error(f"An error occurred: {e}", exc_info=True)


2026-09-03 21:07:35,539 - INFO - Starting MLflow run...
2026-09-03 21:07:36,123 - INFO - Logging preprocessing parameters...
2026-09-03 21:07:37,189 - INFO - Initializing Logistic Regression model...
2026-09-03 21:07:37,192 - INFO - Fitting the model...
2026-09-03 21:07:37,254 - INFO - Model training complete.
2026-09-03 21:07:37,255 - INFO - Logging model parameters...
2026-09-03 21:07:37,618 - INFO - Making predictions...
2026-09-03 21:07:37,630 - INFO - Calculating evaluation metrics...
2026-09-03 21:07:37,643 - INFO - Logging evaluation metrics...
2026-09-03 21:07:38,997 - INFO - Saving and logging the model...
2026/09/03 21:07:38 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026-09-03 21:08:13,806 - INFO - Model training and logging completed in 37.68 seconds.
2026-09-03 21:08:13,808 - INFO - Accuracy: 0.624
2026-09-03 21:08:13,809 - INFO - Precision: 0.6779661016949152
2026-09-03 21:08:13,811 - INFO - Recall: 0.5882352941176471
2026-09-03

🏃 View run intrigued-hound-414 at: https://dagshub.com/tripathianish12/NLP_Sentiment_Analysis_IMDB_reviews.mlflow/#/experiments/0/runs/3e60e1a19c034642bb3bf2d179d4b277
🧪 View experiment at: https://dagshub.com/tripathianish12/NLP_Sentiment_Analysis_IMDB_reviews.mlflow/#/experiments/0
